# Задание 1.
Реализуйте базовый класс Account, который моделирует поведение банковского счёта. Этот класс должен не только выполнять базовые операции, но и вести детальный учёт всех действий, а также предоставлять аналитику по истории операций.

## Этап 1. Реализация базового класса Checking Account
Класс должен быть инициализирован с параметрами:
- account_holder (str) — имя владельца счёта;
- balance (float, по умолчанию 0) — начальный баланс счёта, не может быть отрицательным.

Атрибуты:
- _account_counter — приватный атрибут для хранения количества созданных счетов. Отсчет начинается с 1000;
- holder — хранит имя владельца;
- Checking_account_number — хранит номер счёта;
- _balance — приватный атрибут для хранения текущего баланса;
- operations_history — список или другая структура для хранения истории операций.

> ! Важно: каждая операция должна храниться не просто как число, а как структурированная информация, например, словарь, кортеж или класс. Минимальный набор данных для операции: тип операции (‘deposit’ или ‘withdraw’), сумма, дата и время операции, текущий баланс после операции, статус (‘success’ или ‘fail’).



In [1]:
import csv
import json
import re
from dataclasses import dataclass
from datetime import UTC, datetime
from enum import StrEnum
from pathlib import Path


class OperationType(StrEnum):
    DEPOSIT = 'deposit'
    WITHDRAW = 'withdraw'
    INTEREST = 'interest'


class OperationStatus(StrEnum):
    SUCCESS = 'success'
    FAIL = 'fail'


@dataclass
class Operation:
    timestamp: datetime
    optype: OperationType
    status: OperationStatus
    amount: float
    balance_after: float

    def __str__(self) -> str:
        return (
            f"{self.timestamp:%d.%m.%Y %H:%M:%S} | {self.optype} | "
            f"{self.amount:.2f} | баланс: {self.balance_after:.2f} | "
            f"{self.status}"
        )


class Account:
    _account_counter = 1000
    _NAME_PATTERN = r'(?:[A-Z][a-z]+|[А-ЯЁ][а-яё]+)'
    _HOLDER_PATTERN = re.compile(rf'{_NAME_PATTERN} {_NAME_PATTERN}')
    _DATE_FORMATS = ('%Y-%m-%d %H:%M:%S', '%d/%m/%Y %H:%M')
    allowed_operations = frozenset(
        {OperationType.DEPOSIT, OperationType.WITHDRAW}
    )

    def __init__(self, account_holder: str = 'John Doe',
                 balance: float = 0.0):
        if not self._HOLDER_PATTERN.fullmatch(account_holder):
            raise ValueError(
                'Имя владельца должно быть в формате «Имя Фамилия» '
                'с заглавных букв'
            )
        if balance < 0:
            raise ValueError('Начальный баланс не может быть отрицательным')
        self.holder = account_holder
        self._balance = balance
        Account._account_counter += 1
        self.account_number = f'ACC-{Account._account_counter}'
        self.operations_history = []

    def _record(self, optype: OperationType, status: OperationStatus,
                amount: float):
        """Добавляет операцию в историю с текущим временем и балансом."""
        self.operations_history.append(
            Operation(datetime.now(UTC), optype, status, amount, self._balance)
        )

    def deposit(self, amount: float):
        """Пополняет счёт и записывает операцию в историю.

        Args:
            amount: Сумма пополнения, должна быть положительной.

        Raises:
            ValueError: Если сумма не положительная.
        """
        if (amount < 0):
            raise ValueError('Сумма должна быть положительной')
        self._balance += amount
        self._record(OperationType.DEPOSIT, OperationStatus.SUCCESS, amount)

    def withdraw(self, amount):
        """Снимает средства со счёта и записывает операцию в историю.

        Args:
            amount: Сумма снятия, должна быть положительной.
        """
        if (amount < 0):
            raise ValueError('Сумма должна быть положительной')
        if (self._balance < amount):
            self._record(OperationType.WITHDRAW, OperationStatus.FAIL, amount)
        else:
            self._balance -= amount
            self._record(
                OperationType.WITHDRAW, OperationStatus.SUCCESS, amount
            )

    def get_balance(self) -> float:
        return self._balance

    def get_history(self) -> list:
        return self.operations_history

    def show_top_operations(
        self,
        n: int = 5,
        min_amount: float = 0,
        since: datetime | None = None,
    ) -> list[Operation]:
        """Выводит и возвращает n крупных успешных операций с фильтрами.

        Сначала операции фильтруются по сумме и дате, затем сортируются
        по сумме по убыванию (при равных суммах выше более поздние)
        и берутся первые n.

        Args:
            n: Сколько операций показать.
            min_amount: Минимальная сумма операции.
            since: Только операции не раньше этой даты (с часовым поясом).
        """
        ops = [
            op for op in self.operations_history
            if op.status == OperationStatus.SUCCESS
            and op.amount >= min_amount
            and (since is None or op.timestamp >= since)
        ]
        top = sorted(
            ops, key=lambda op: (op.amount, op.timestamp), reverse=True
        )[:n]
        print(*top, sep='\n')
        return top

    @staticmethod
    def _read_file(path: str) -> list[dict]:
        """Читает транзакции из .csv или .json и возвращает список словарей."""
        suffix = Path(path).suffix.lower()
        if suffix not in ('.csv', '.json'):
            raise ValueError(f'Неподдерживаемый формат файла: {suffix}')
        with open(path, encoding='utf-8', newline='') as f:
            if suffix == '.json':
                return json.load(f)
            return list(csv.DictReader(f))

    @classmethod
    def _parse_date(cls, value: str) -> datetime:
        """Разбирает дату в одном из известных форматов."""
        for fmt in cls._DATE_FORMATS:
            try:
                return datetime.strptime(value, fmt).replace(tzinfo=UTC)
            except ValueError:
                continue
        raise ValueError(f'Неверный формат даты: {value!r}')

    @classmethod
    def _parse_record(cls, record: dict) -> Operation | None:
        """Возвращает операцию для валидной записи, иначе None."""
        try:
            operation = Operation(
                timestamp=cls._parse_date(record['date']),
                optype=OperationType(record['operation']),
                status=OperationStatus(record['status']),
                amount=float(record['amount']),
                balance_after=float(record['balance_after']),
            )
        except (KeyError, TypeError, ValueError):
            return None
        if (
            operation.amount <= 0
            or operation.optype not in cls.allowed_operations
        ):
            return None
        return operation

    @classmethod
    def clean_history(cls, records: list[dict]) -> list[Operation]:
        """Отбирает валидные записи: без опечаток, дублей, неверных данных.

        Args:
            records: Сырые записи из файла (словари).

        Returns:
            Валидные операции в хронологическом порядке.
        """
        seen = set()
        operations = []
        for record in records:
            key = tuple(record.items())
            if key in seen:
                continue
            seen.add(key)
            operation = cls._parse_record(record)
            if operation is not None:
                operations.append(operation)
        return sorted(operations, key=lambda op: op.timestamp)

    def load_history(self, path: str) -> int:
        """Загружает операции этого счёта из CSV/JSON и обновляет баланс.

        Из общего файла берутся только записи с номером этого счёта.
        Баланс пересчитывается: к текущему балансу счёта добавляются все
        загруженные успешные операции, balance_after считается заново.

        Args:
            path: Путь к файлу .csv или .json с транзакциями всех счетов.

        Returns:
            Количество загруженных операций.
        """
        records = [
            r for r in self._read_file(path)
            if r.get('account_number') == self.account_number
        ]
        operations = self.clean_history(records)
        for operation in operations:
            if operation.status == OperationStatus.SUCCESS:
                sign = -1 if operation.optype == OperationType.WITHDRAW else 1
                self._balance += sign * operation.amount
            operation.balance_after = self._balance
            self.operations_history.append(operation)
        return len(operations)


acc = Account('Георгий Лялин', 120)

try:
    acc.deposit(80)
    acc.deposit(-80)
except ValueError as err:
    print(err)

try:
    acc.withdraw(200)
    acc.withdraw(200)
    acc.withdraw(-200)
except ValueError as err:
    print(err)
print(*acc.operations_history, sep='\n')


Сумма должна быть положительной
Сумма должна быть положительной
04.10.2026 11:26:20 | deposit | 80.00 | баланс: 200.00 | success
04.10.2026 11:26:20 | withdraw | 200.00 | баланс: 0.00 | success
04.10.2026 11:26:20 | withdraw | 200.00 | баланс: 0.00 | fail


# Задание 2.
Реализуйте два класса CheckingAccount (расчётный счёт) и SavingsAccount (сберегательный счёт), которые отражают абстракцию базового поведения банковских аккаунтов:
- наследуются от базового класса Account;
- хранят атрибут класса account_type.

Класс SavingsAccount (сберегательный счёт) дополнительно должен реализовывать метод расчёта процентов на остаток apply_interest(self, rate) (например, 7% на остаток).
Класс SavingsAccount (сберегательный счёт) позволяет снимать деньги только до определенного порога баланса: нельзя снять больше 50% от баланса. Переопределите метод снятия со счёта.
Реализуйте валидацию на отрицательные суммы и корректность имени владельца:
- имя владельца счёта должно быть в формате «Имя Фамилия» с заглавных букв, кириллицей или латиницей, иначе — должно вызываться исключение;
- попытка положить отрицательную сумму должна вызывать исключение.
Реализуйте метод для анализа истории транзакций по размеру и дате:
- метод должен выводить последние n крупных операций.

In [2]:
class CheckingAccount(Account):
    """Расчётный счёт"""

    account_type = 'checking'


class SavingsAccount(Account):
    """Сберегательный счёт: проценты на остаток и лимит на снятие."""

    account_type = 'savings'
    MAX_WITHDRAW_SHARE = 0.5
    allowed_operations = Account.allowed_operations | {OperationType.INTEREST}

    def apply_interest(self, rate: float) -> float:
        """Начисляет проценты на остаток и записывает операцию в историю.

        Args:
            rate: Процентная ставка в процентах.

        Returns:
            Сумма начисленных процентов.

        Raises:
            ValueError: Если ставка отрицательная.
        """
        if rate < 0:
            raise ValueError('Процентная ставка не может быть отрицательной')
        interest = self._balance * rate / 100
        self._balance += interest
        self._record(
            OperationType.INTEREST, OperationStatus.SUCCESS, interest
        )
        return interest

    def withdraw(self, amount: float):
        """Снимает средства, но не больше 50% от текущего баланса.

        При превышении лимита операция записывается со статусом fail.

        Args:
            amount: Сумма снятия, должна быть положительной.

        Raises:
            ValueError: Если сумма отрицательная.
        """
        if amount > self._balance * self.MAX_WITHDRAW_SHARE:
            self._record(OperationType.WITHDRAW, OperationStatus.FAIL, amount)
            return
        super().withdraw(amount)


# Проверка
check = CheckingAccount('Иван Петров', 1000)
check.deposit(500)
check.withdraw(300)

save = SavingsAccount('Anna Smith', 1000)
save.apply_interest(7)      # +70 -> 1070
save.withdraw(600)          # больше 50% (535) -> fail
save.withdraw(500)          # допустимо -> success
save.deposit(2000)

print(check.account_type, check.get_balance())
print(save.account_type, save.get_balance())
print(*save.get_history(), sep='\n')

print('Топ-3 операции:')
save.show_top_operations(3)

for bad_name in ('иван петров', 'Иван', 'Ivan  Petrov', 'Иван Petrov1'):
    try:
        CheckingAccount(bad_name)
    except ValueError as err:
        print(f'{bad_name!r}: {err}')

try:
    save.deposit(-10)
except ValueError as err:
    print(err)


checking 1200
savings 2570.0
04.10.2026 11:26:20 | interest | 70.00 | баланс: 1070.00 | success
04.10.2026 11:26:20 | withdraw | 600.00 | баланс: 1070.00 | fail
04.10.2026 11:26:20 | withdraw | 500.00 | баланс: 570.00 | success
04.10.2026 11:26:20 | deposit | 2000.00 | баланс: 2570.00 | success
Топ-3 операции:
04.10.2026 11:26:20 | deposit | 2000.00 | баланс: 2570.00 | success
04.10.2026 11:26:20 | withdraw | 500.00 | баланс: 570.00 | success
04.10.2026 11:26:20 | interest | 70.00 | баланс: 1070.00 | success
'иван петров': Имя владельца должно быть в формате «Имя Фамилия» с заглавных букв
'Иван': Имя владельца должно быть в формате «Имя Фамилия» с заглавных букв
'Ivan  Petrov': Имя владельца должно быть в формате «Имя Фамилия» с заглавных букв
'Иван Petrov1': Имя владельца должно быть в формате «Имя Фамилия» с заглавных букв
Сумма должна быть положительной


# Задание 3.
Перед началом работы загрузите из личного кабинете файлы, на которых можно проверить код с «грязными» данными: transactions_dirty.csv и transactions_dirty.json.
Реализуйте для классов аккаунтов CheckingAccount (расчётный счёт) и SavingsAccount (сберегательный счёт) два метода:
- Метод загрузки истории в аккаунт из файла с транзакциями (файл транзакций общий для всех аккаунтов, необходимо учесть фильтрацию загружаемых значений).
- Метод clean_history(), который ищет ошибки в данных перед записью транзакций в историю (опечатки, отрицательные суммы, неверные даты). Обратите внимание, что для SavingsAccount (сберегательный счёт) доступно три типа операции (deposit, withdraw и interest), в то время как для CheckingAccount (расчётный счёт) доступны только два типа операции (deposit, withdraw). Все данные с ошибками считаем невалидными и не записываем в историю операций.
- После загрузки истории операций в аккаунт, баланс счёта должен обновиться.

In [3]:

with open('transactions_dirty.csv', encoding='utf-8', newline='') as f:
    reader = csv.DictReader(f)
    for row in reader:
        print(row['account_number'], row['amount'])

ACC-100001 921.0
ACC-100001 607.0
ACC-100001 488.0
ACC-100001 129.0
ACC-100001 880.0
ACC-100001 
ACC-100001 352.0
ACC-100001 65.0
ACC-100001 654.0
ACC-100001 245.0
ACC-100001 532.0
ACC-100001 327.0
ACC-100001 333.0
ACC-100001 793.0
ACC-100001 198.0
ACC-100001 669.0
ACC-100001 744.0
ACC-100001 111.0
ACC-100001 838.0
ACC-100001 532.0
ACC-100001 393.0
ACC-100001 701.0
ACC-100001 308.0
ACC-100001 135.0
ACC-100001 916.0
ACC-100002 469.0
ACC-100002 392.0
ACC-100002 317.0
ACC-100002 797.0
ACC-100002 
ACC-100002 840.0
ACC-100002 183.28
ACC-100002 192.44
ACC-100002 -136.0
ACC-100002 426.0
ACC-100002 638.0
ACC-100002 170.0
ACC-100002 432.0
ACC-100002 622.0
ACC-100002 306.26
ACC-100002 278.0
ACC-100002 440.0
ACC-100002 231.0
ACC-100002 274.13
ACC-100002 -872.0
ACC-100002 286.0
ACC-100002 300.0
ACC-100002 302.13
ACC-100002 457.0
ACC-100002 282.0
ACC-100003 233.0
ACC-100003 292.0
ACC-100003 720.0
ACC-100003 290.0
ACC-100003 296.0
ACC-100003 956.0
ACC-100003 754.0
ACC-100003 255.0
ACC-100003 -115.0


In [4]:
cases = [
    (CheckingAccount, 'Иван Петров', 'ACC-100001', 1200),
    (SavingsAccount, 'Anna Smith', 'ACC-100002', 1500),
    (CheckingAccount, 'Пётр Сидоров', 'ACC-100003', 800),
    (SavingsAccount, 'Maria Ivanova', 'ACC-100004', 2000),
]

for path in ('transactions_dirty.json', 'transactions_dirty.csv'):
    print(f'{path}')
    for account_class, holder, number, start_balance in cases:
        account = account_class(holder, start_balance)
        account.account_number = number
        records = Account._read_file(path)
        total = sum(r['account_number'] == number for r in records)
        loaded = account.load_history(path)
        print(
            f'{number} {account_class.__name__}: '
            f'загружено {loaded} из {total}, '
            f'баланс {start_balance} -> {account.get_balance():.2f}'
        )


transactions_dirty.json
ACC-100001 CheckingAccount: загружено 17 из 26, баланс 1200 -> 4864.00
ACC-100002 SavingsAccount: загружено 18 из 27, баланс 1500 -> 3991.24
ACC-100003 CheckingAccount: загружено 13 из 21, баланс 800 -> 5446.00
ACC-100004 SavingsAccount: загружено 19 из 31, баланс 2000 -> 10057.25
transactions_dirty.csv
ACC-100001 CheckingAccount: загружено 17 из 26, баланс 1200 -> 4864.00
ACC-100002 SavingsAccount: загружено 18 из 27, баланс 1500 -> 3991.24
ACC-100003 CheckingAccount: загружено 13 из 21, баланс 800 -> 5446.00
ACC-100004 SavingsAccount: загружено 19 из 31, баланс 2000 -> 10057.25
